In [5]:
#!/usr/bin/env python3
"""
volume_to_weight_train_bayes.py

Full pipeline with Bayesian Optimization:
 - Input: CSV mapping (image_path, weight) and image root folder.
 - Precompute volumes via Otsu threshold + bounding box + slice approximation.
 - Train polynomial regression (PyTorch) with elastic-net penalty.
 - Hyperparameters tuned by Bayesian optimization (lr, alpha, l1_ratio, degree, N_slices).
 - Supports GPU acceleration if available.
"""

import os
import math
import pickle
import time
from typing import List, Tuple

import numpy as np
from PIL import Image
from tqdm import tqdm
import csv
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args

# ----------------------------
# Utilities: Otsu thresholding
# ----------------------------
def otsu_threshold_from_array(gray: np.ndarray) -> int:
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256))
    total = hist.sum()
    if total == 0:
        return 0
    prob = hist.astype(np.float64) / total
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * np.arange(256))
    mu_total = mu[-1]
    denom = omega * (1.0 - omega) + 1e-12
    sigma_b2 = (mu_total * omega - mu) ** 2 / denom
    sigma_b2[omega == 0] = 0
    sigma_b2[omega == 1] = 0
    return int(np.argmax(sigma_b2))

# ----------------------------
# Mask / bbox / volume utils
# ----------------------------
def image_to_mask(img_path: str) -> np.ndarray:
    img = Image.open(img_path).convert("L")
    arr = np.array(img)
    t = otsu_threshold_from_array(arr)
    mask = (arr > t).astype(np.uint8)
    return mask

def bbox_from_mask(mask: np.ndarray) -> Tuple[int,int,int,int]:
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return (0,0,0,0)
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def avg_row_width(mask: np.ndarray, y: int, minx: int, maxx: int, half_window: int = 1) -> float:
    H, W = mask.shape
    ys = range(max(0, y-half_window), min(H, y+half_window+1))
    widths = [int(np.sum(mask[yy, minx:maxx+1] > 0)) for yy in ys]
    return float(np.mean(widths)) if widths else 0.0

def estimate_volume_from_mask(mask: np.ndarray, N_slices: int, bbox: Tuple[int,int,int,int], boundary_window: int = 1) -> float:
    minx, miny, maxx, maxy = bbox
    if minx == maxx and miny == maxy:
        return 0.0

    L = maxy - miny + 1
    slice_h = float(L) / float(N_slices)

    def area_at_row(yf: float) -> float:
        y = int(round(yf))
        y = max(miny, min(maxy, y))
        w = avg_row_width(mask, y, minx, maxx, boundary_window)
        return (math.pi / 4.0) * (w ** 2)

    total_vol = 0.0
    for i in range(N_slices):
        top_f = miny + i * slice_h
        bottom_f = miny + (i + 1) * slice_h
        h = bottom_f - top_f
        A_top = area_at_row(top_f)
        A_bottom = area_at_row(bottom_f)
        if i == 0:
            V = (h / 3.0) * A_bottom
        elif i == N_slices - 1:
            V = (h / 3.0) * A_top
        else:
            V = (h / 3.0) * (A_top + A_bottom + math.sqrt(max(0.0, A_top * A_bottom)))
        total_vol += V
    return total_vol

# ----------------------------
# CSV loader
# ----------------------------
def load_csv_pairs(csv_file: str, img_root: str = "") -> List[Tuple[str, float]]:
    """
    Load (image_path, weight) pairs from CSV. Resolve each image path robustly:
      - if path is absolute and exists -> use it
      - else if joining with img_root exists -> use that
      - else if path already contains img_root -> normalize and try it
      - else try basename join with img_root
    If file does not exist after attempts, skip that row and warn.
    """
    pairs = []
    missing = []
    img_root_norm = os.path.normpath(img_root) if img_root else ""

    with open(csv_file, "r", newline='') as f:
        reader = csv.reader(f)
        first = next(reader, None)
        header = False
        if first and len(first) >= 2:
            if "image" in first[0].lower() or "weight" in first[1].lower():
                header = True
        # helper
        def resolve_path(pth: str) -> str:
            if not pth:
                return ""
            pth = pth.strip().strip('"').strip("'")
            pth_norm = os.path.normpath(pth)
            # 1) absolute path as given
            if os.path.isabs(pth_norm) and os.path.exists(pth_norm):
                return pth_norm
            # 2) if pth already contains the img_root substring, try normalizing and using it
            if img_root_norm and img_root_norm in pth_norm:
                cand = os.path.normpath(pth_norm)
                if os.path.exists(cand):
                    return cand
                # try removing duplicate prefix (e.g., img_root + img_rel)
                try_rel = pth_norm.split(img_root_norm, 1)[-1].lstrip(os.sep)
                cand2 = os.path.join(img_root_norm, try_rel)
                if os.path.exists(cand2):
                    return cand2
            # 3) join with img_root
            if img_root_norm:
                cand = os.path.join(img_root_norm, pth_norm)
                if os.path.exists(cand):
                    return cand
            # 4) try just basename in img_root
            b = os.path.basename(pth_norm)
            if img_root_norm:
                cand = os.path.join(img_root_norm, b)
                if os.path.exists(cand):
                    return cand
            # 5) finally, if relative path exists as-is relative to CWD
            if os.path.exists(pth_norm):
                return pth_norm
            return ""  # unresolved

        # If first row wasn't a header, process it
        if first and not header:
            try:
                img_p = resolve_path(first[0])
                if img_p:
                    pairs.append((img_p, float(first[1])))
                else:
                    missing.append(first[0])
            except Exception:
                missing.append(first[0] if first else "<unknown>")

        for row in reader:
            if len(row) < 2:
                continue
            img_raw = row[0]
            try:
                wt = float(row[1])
            except:
                # try stripping non-numeric characters, otherwise skip
                try:
                    wt = float(row[1].strip())
                except:
                    missing.append(img_raw)
                    continue
            img_p = resolve_path(img_raw)
            if img_p:
                pairs.append((img_p, wt))
            else:
                missing.append(img_raw)

    if missing:
        print(f"[warning] {len(missing)} image paths in CSV could not be resolved - they were skipped.")
        # Optionally show a few examples:
        for m in missing[:10]:
            print("  -", m)
    print(f"[info] Loaded {len(pairs)} valid image pairs from '{csv_file}'.")
    return pairs


# ----------------------------
# Features
# ----------------------------
class SimpleScaler:
    def fit(self, X): self.mean_, self.std_ = np.mean(X,0,keepdims=True), np.std(X,0,keepdims=True); self.std_[self.std_==0]=1
    def transform(self, X): return (X - self.mean_) / self.std_
    def fit_transform(self, X): self.fit(X); return self.transform(X)

def poly_features(vols: np.ndarray, degree: int) -> np.ndarray:
    return np.column_stack([vols[:,0]**d for d in range(1, degree+1)])

# ----------------------------
# Model
# ----------------------------
class PolyRegModel(nn.Module):
    def __init__(self, in_features): super().__init__(); self.lin = nn.Linear(in_features, 1)
    def forward(self, x): return self.lin(x).squeeze(1)

# ----------------------------
# Train utils
# ----------------------------
def train_one_model(X_train, y_train, X_val, y_val, degree, lr, alpha, l1_ratio, batch_size, epochs, device):
    X_train_f = poly_features(X_train.reshape(-1,1), degree)
    X_val_f = poly_features(X_val.reshape(-1,1), degree)
    scaler = SimpleScaler()
    X_train_s = scaler.fit_transform(X_train_f)
    X_val_s = scaler.transform(X_val_f)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_train_s,dtype=torch.float32), torch.tensor(y_train,dtype=torch.float32)),
                              batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.tensor(X_val_s,dtype=torch.float32), torch.tensor(y_val,dtype=torch.float32)),
                            batch_size=max(1,batch_size//2), shuffle=False)

    model = PolyRegModel(degree).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state = float('inf'), None
    for _ in range(epochs):
        model.train()
        for xb,yb in train_loader:
            xb,yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            preds = model(xb)
            mse = ((preds - yb)**2).mean()
            l1 = sum(p.abs().sum() for p in model.parameters())
            l2 = sum((p**2).sum() for p in model.parameters())
            loss = mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = 0
            for xb,yb in val_loader:
                xb,yb = xb.to(device), yb.to(device)
                preds = model(xb)
                mse = ((preds - yb)**2).mean()
                l1 = sum(p.abs().sum() for p in model.parameters())
                l2 = sum((p**2).sum() for p in model.parameters())
                val_loss += (mse + alpha*(l1_ratio*l1 + (1-l1_ratio)*0.5*l2)).item()
            val_loss /= len(val_loader)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return model, scaler, best_loss

# ----------------------------
# Globals for Bayesian Opt
# ----------------------------
_GLOBALS = {}
search_space = []

def _bayes_obj_list(x):
    # x is a list: [lr, alpha, l1_ratio, degree, N_slices]
    lr, alpha, l1_ratio, degree, N_slices = x
    lr = float(lr); alpha = float(alpha); l1_ratio = float(l1_ratio)
    degree = int(degree); N_slices = int(N_slices)

    G = _GLOBALS
    vols = []
    # Compute volumes for the *whole* dataset so we can index with absolute indices
    for p in G["pairs"]:
        try:
            mask = image_to_mask(p[0])
            bbox = bbox_from_mask(mask)
            vol = estimate_volume_from_mask(mask, N_slices, bbox)
            vols.append(vol)
        except:
            vols.append(0.0)
    vols = np.array(vols).reshape(-1, 1)

    train_idx = G["train_idx"]
    val_idx = G["val_idx"]

    Xtr, ytr = vols[train_idx], G["weights"][train_idx]
    Xv, yv = vols[val_idx], G["weights"][val_idx]

    _, _, vloss = train_one_model(Xtr, ytr, Xv, yv, degree, lr, alpha, l1_ratio,
                                  G["batch_size"], G["inner_epochs"], G["device"])
    return vloss

# ----------------------------
# Pipeline
# ----------------------------
def run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max, **kwargs):
    pairs = load_csv_pairs(csv_file, img_root)
    weights = np.array([w for _,w in pairs])
    n = len(pairs)
    idx = np.arange(n); np.random.shuffle(idx)
    val_n, test_n = max(1,int(0.2*n)), max(1,int(0.1*n))
    val_idx, test_idx, train_idx = idx[:val_n], idx[val_n:val_n+test_n], idx[val_n+test_n:]

    # Store globals
    _GLOBALS.update({
        "pairs": pairs,                 # full list of (img_path, weight)
        "weights": weights,             # full weights array
        "train_idx": train_idx,         # absolute indices into pairs/weights
        "val_idx": val_idx,             # absolute indices into pairs/weights
        "batch_size": kwargs["batch_size"],
        "inner_epochs": kwargs["bayes_inner_epochs"],
        "device": kwargs["device"]
    })

    global search_space
    search_space = [
        Real(1e-5,1e-1,"log-uniform",name="lr"),
        Real(1e-8,1e-2,"log-uniform",name="alpha"),
        Real(0.0,1.0,name="l1_ratio"),
        Integer(1,6,name="degree"),
        Integer(N_slices_min,N_slices_max,name="N_slices")
    ]

    res = gp_minimize(_bayes_obj_list, search_space,n_calls=kwargs["bayes_calls"],n_initial_points=kwargs["bayes_init_points"],random_state=42)
    best = dict(zip(["lr","alpha","l1_ratio","degree","N_slices"],res.x))
    print("Best hyperparams:", best)

    # Final training uses full train+val
    vols = []
    for p in pairs:
        mask = image_to_mask(p[0]); bbox = bbox_from_mask(mask)
        vols.append(estimate_volume_from_mask(mask, best["N_slices"], bbox))
    vols = np.array(vols).reshape(-1,1)
    Xtr = np.delete(vols,val_idx,axis=0); ytr = np.delete(weights,val_idx)
    Xte, yte = vols[test_idx], weights[test_idx]

    model,scaler,_ = train_one_model(Xtr,ytr,Xte,yte,int(best["degree"]),best["lr"],best["alpha"],best["l1_ratio"],
                                     kwargs["batch_size"],kwargs["epochs"],kwargs["device"])
    torch.save({"state":model.state_dict(),"scaler_mean":scaler.mean_,"scaler_std":scaler.std_,"params":best}, out_model)
    print(f"Model saved to {out_model}")

# ----------------------------
# Hard-coded config (EDIT THESE)
# ----------------------------
if __name__=="__main__":
    # === EDIT THESE ===
    img_root = "../Tomato-Yield-Estimation-ML-DL-Approaches/images"   # folder containing images referenced in CSV (or full paths in CSV)
    csv_file = "../Tomato-Yield-Estimation-ML-DL-Approaches/image_weights.csv"  # CSV with (image_relative_path, weight) rows
    out_model = "tomato_volume_model.pth"   # where to save final model
    N_slices_min = 10
    N_slices_max = 15

    # Training / bayes settings
    epochs = 50
    batch_size = 8
    bayes_calls = 12
    bayes_init_points = 4
    bayes_inner_epochs = 8

    # device: set to "cpu", "cuda", or "auto"
    device = "auto"
    # === END EDITS ===

    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    run_pipeline(img_root, csv_file, out_model, N_slices_min, N_slices_max,
                 epochs=epochs, batch_size=batch_size,
                 bayes_calls=bayes_calls, bayes_init_points=bayes_init_points,
                 bayes_inner_epochs=bayes_inner_epochs, device=device)


[info] Loaded 7224 valid image pairs from '../Tomato-Yield-Estimation-ML-DL-Approaches/image_weights.csv'.
Best hyperparams: {'lr': 0.002950706670790534, 'alpha': 4.676478725076053e-05, 'l1_ratio': 0.007066305219717408, 'degree': 1, 'N_slices': 13}
Model saved to tomato_volume_model.pth
